## RDKit Fingerprints

https://www.rdkit.org/docs/GettingStartedInPython.html

In [35]:
import pickle
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import DataStructs
import numpy as np

In [4]:
with open("../data/final_drugs_df","rb") as f:
    final_drugs_df =pickle.load(f)
with open("../data/final_indications_df","rb") as f:
    final_indications_df = pickle.load(f)
with open("../data/final_diseases_df","rb") as f:
    final_diseases_df =pickle.load(f)
with open("../data/final_trials_df","rb") as f:
    final_trials_df =pickle.load(f)

In [54]:
fpgen = AllChem.GetMorganGenerator(radius=2)
mol = Chem.MolFromSmiles('Cc1ccccc1')
fp = fpgen.GetFingerprint(mol)
bitstring = fp.ToBitString()  
arr = np.fromiter(bitstring, dtype=int)
arr

array([0, 0, 0, ..., 0, 0, 0], shape=(2048,))

In [65]:
final_drugs_df = final_drugs_df[ final_drugs_df["canonical_smiles"].notna() ]

In [69]:
# Source - https://stackoverflow.com/a
# Posted by let me down slowly
# Retrieved 2025-11-21, License - CC BY-SA 4.0

smiles_list = final_drugs_df["canonical_smiles"]#["O=C(NCc1cc(OC)c(O)cc1)CCCC/C=C/C(C)C", "CC(C)CCCCCC(=O)NCC1=CC(=C(C=C1)O)OC", "c1(C=O)cc(OC)c(O)cc1"]

# create a list of mols
mols = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]

# create a list of fingerprints from mols
fps = np.array([np.fromiter(Chem.RDKFingerprint(mol).ToBitString(), dtype=int)  for mol in mols])


In [75]:
from sklearn.decomposition import PCA
import pandas as pd

pca = PCA(n_components=10)
X_reduced = pca.fit_transform(fps)
fp_df = pd.DataFrame(X_reduced, columns=[f'FP_{i+1}' for i in range(X_reduced.shape[1])])

print(X_reduced.shape)  # (num_molecules, 10)

(418, 10)


In [76]:
final_drugs_df = pd.concat([final_drugs_df.reset_index(drop=True), fp_df], axis=1)

In [77]:
final_drugs_df

,drug_id,drug_name,biotherapeutic,black_box_warning,chemical_probe,chirality,dosed_ingredient,first_approval,first_in_class,helm_notation,...,FP_1,FP_2,FP_3,FP_4,FP_5,FP_6,FP_7,FP_8,FP_9,FP_10
0,CHEMBL6,indomethacin,0,1,0,achiral,True,1965.0,0,None,...,7.888383,-2.577735,-1.848938,1.827325,2.893642,-4.821288,4.631789,-0.676535,2.032592,-1.310267
1,CHEMBL8,ciprofloxacin,0,1,0,achiral,True,1987.0,0,None,...,7.188543,-1.363429,-3.352934,-1.907561,2.565275,-3.262328,2.888717,0.678062,4.459360,-1.318801
2,CHEMBL12,diazepam,0,1,0,achiral,True,1963.0,0,None,...,-0.435942,-1.553639,-2.879754,-1.183526,0.365185,2.030808,-1.980848,-1.403077,4.587439,1.504602
3,CHEMBL413,sirolimus,0,1,1,single_enantiomer,True,1999.0,0,None,...,7.481606,7.506252,5.782022,-1.172454,-5.316319,2.168579,1.055328,7.213992,2.711400,-0.868101
4,CHEMBL269732,tacrolimus anhydrous,0,1,1,single_enantiomer,False,1994.0,0,None,...,8.188933,7.424781,5.508912,-0.774401,-6.318303,2.038989,2.314694,7.675512,3.229301,-1.230261
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,CHEMBL6067994,vorbipiprant,0,0,0,single_enantiomer,False,NaN,0,None,...,1.696256,-4.314272,5.852710,0.739827,1.760976,5.307039,1.455400,1.733640,-3.085049,-2.673799
414,CHEMBL6067997,apilimod mesylate,0,0,0,achiral,False,NaN,0,None,...,-4.487374,-1.139341,-3.483671,-1.216811,-1.173518,-2.512167,-3.008633,-0.185777,-1.730143,-0.751057
415,CHEMBL6068328,eltrombopag choline,0,1,0,achiral,True,2023.0,0,None,...,7.366967,-0.416525,-5.176704,0.093746,1.499170,5.350981,-8.339791,-4.881114,8.758576,4.806281
416,CHEMBL6068392,icovamenib,0,0,0,single_enantiomer,False,NaN,0,None,...,10.521086,-1.555111,-0.729076,-5.089454,1.971648,3.401369,-2.371107,3.629812,-2.462524,-1.013249
